# **Семинар 4: Эмбеддинги и нейронные сети**
## Содержание занятия:

### Тема 1. Векторизация с использованием эмбеддингов
Рассматриваемые вопросы:
*   **Векторизация текста:** Преобразование текста в числовые векторы для анализа.
*   **N-граммы:** Анализ последовательностей слов для понимания контекста.
*   **Языковые модели:** Построение моделей для предсказания следующего слова в последовательности. Простая языковая модель (биграммы)   
*   **Векторные представления слов (Embeddings):**
    *   Word2Vec
    *   FastText
    *   GloVe
    *   Работа с эмбеддингами
*   **Эмбеддинги для русскоязычных текстов** на примере библиотеки Navec
*   **Косинусное сходство/расстояние**

### Тема 2. Основы нейронных сетей - градиентный спуск
Рассматриваемые вопросы:

*   Функции активации
*   Функции потерь
*   Градиентный спуск
*   SGD и его модификации (Momentum, Adam и др.)







In [ ]:
!pip install pymorphy3

In [ ]:
!pip install gensim

In [ ]:
!pip install navec

In [ ]:
# Импорт необходимых библиотек
# Успешное выполнение этой ячейки кода подтверждает правильную настройку среды разработки

import numpy as np

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

import re
from collections import defaultdict
import pymorphy3

import gensim.downloader as api

from navec import Navec
import requests

from scipy.spatial.distance import cosine

from matplotlib import pyplot as plt

## Векторизация текста и N-граммы

На предыдущих практических занятиях и лекциях мы уже сталкивались с векторизацией текста, но пока что использовали только простые методы, не учитывающие при построении векторов контекст и порядок слов (One-hot encoding, Bag-of-Words (Мешок слов), TF-IDF). Теперь мы на практике разберём подходы, которые учитывают эти особенности в тексте.

Начнём со статистических моделей векторизации.

### **N-граммы**

**N-граммы** - это последовательности из N элементов (в нашем случае - слов). Использование N-грамм при векторизации позволяет учесть некоторый ограниченный контекст и порядок слов. Например, биграммы (N=2) учитывают пары соседних слов, а триграммы (N=3) - тройки соседних слов (см. слайды Лекции 3). Включение N-грамм в модель векторизации может улучшить качество анализа, особенно для задач, где важен порядок слов (например, анализ тональности или машинный перевод).


### Задание 1: Векторизация и N-граммы
В этом задании вам предстоит поработать с N-граммами, используя классы из `sklearn.feature_extraction.text`.

Используя `CountVectorizer` или `TfidfVectorizer` с параметром `ngram_range`, мы можем получить различные представления с использованием N-грамм.

Параметр `ngram_range` задаёт нижнюю и верхнюю границы диапазона значений n для n-грамм, которые необходимо извлечь. Будут использоваться все значения n, такие, что min_n <= n <= max_n. Например, ngram_range из (1, 1) учитывает только униграммы, (1, 2) - униграммы и биграммы, а (2, 2) - только биграммы.

Ниже приведены примеры для разных представлений текстов с `CountVectorizer`. Для каждого полученного представления выведем результирующую матрицу векторизации и список признаков (N-грамм).

In [ ]:
# Создание корпуса текстовых документов (строк)
corpus = [
    "Этот текст - простой пример.",
    "Пример простой векторизации текста.",
    "Векторизация текста - важный шаг."
]

# Биграммы с CountVectorizer
vectorizer_bigrams = CountVectorizer(ngram_range=(2, 2))
X_bigrams = vectorizer_bigrams.fit_transform(corpus)
feature_names_bigrams = vectorizer_bigrams.get_feature_names_out()

print("--- Биграммы (CountVectorizer) ---")
print("Матрица векторизации:")
print(X_bigrams.toarray())
print("Список признаков (биграмм):")
print(feature_names_bigrams,"\n")


# Униграммы и биграммы с CountVectorizer
vectorizer_unigrams_bigrams = CountVectorizer(ngram_range=(1, 2))
X_unigrams_bigrams = vectorizer_unigrams_bigrams.fit_transform(corpus)
feature_names_unigrams_bigrams = vectorizer_unigrams_bigrams.get_feature_names_out()

print("--- Униграммы и Биграммы (CountVectorizer) ---")
print("Матрица векторизации:")
print(X_unigrams_bigrams.toarray())
print("Список признаков (униграмм и биграмм):")
print(feature_names_unigrams_bigrams,"\n")


**Задание:**

Получите представление TF-IDF с использованием биграмм и триграмм

In [ ]:

#### ВСТАВЬТЕ КОД СЮДА
...
####

### Построение простой статистической языковой модели (N-граммы)

Построим и протестируем простую биграммную (N=2) языковую модель.

Проведём предобработку текстовых документов в корпусе: приведение к нормальной форме, удаление пунктуации

In [ ]:
corpus = [
    "Этот текст - простой пример.",
    "Пример простой векторизации текста.",
    "Векторизация текста - важный шаг."
]

# создаём морфологический анализатор
morph = pymorphy3.MorphAnalyzer()

# 2. Предобработка текста (приведение к нормальной форме, удаление пунктуации)
preprocessed_corpus = []
for text in corpus:
    text = re.sub(r'[^\w\s]', '', text) # Удаление пунктуации, кроме пробелов
    # разбиваем на слова
    words = text.split()
    # приводим к леммам
    lemmas = [morph.parse(word)[0].normal_form for word in words]
    text = " ".join(lemmas)
    preprocessed_corpus.append(text)

preprocessed_corpus



Получим отдельно представление с использованием биграмм и предсталение с использованием униграмм (просто отдельных слов).  

In [ ]:
# Биграммы с CountVectorizer
vectorizer_bigrams = CountVectorizer(ngram_range=(2, 2))
X_bigrams = vectorizer_bigrams.fit_transform(preprocessed_corpus)
feature_names_bigrams = vectorizer_bigrams.get_feature_names_out()

print("--- Биграммы (CountVectorizer) ---")
print("Матрица векторизации:")
print(X_bigrams.toarray())
print("Список признаков (биграмм):")
print(feature_names_bigrams,"\n")

# Униграммы с CountVectorizer
vectorizer_unigrams = CountVectorizer(ngram_range=(1, 1))
X_unigrams = vectorizer_unigrams.fit_transform(preprocessed_corpus)
feature_names_unigrams = vectorizer_unigrams.get_feature_names_out()

print("--- Униграммы (CountVectorizer) ---")
print("Матрица векторизации:")
print(X_unigrams.toarray())
print("Список признаков (униграмм):")
print(feature_names_unigrams,"\n")

Просуммируем матрицы векторизации по столбцам и получим вектор частот встречаемости для униграмм и биграмм

In [ ]:
# Подсчёт общей частоты встречаемости униграмм и биграмм во всём корпусе
count_in_corpus_unigrams = X_unigrams.sum(axis=0)
count_in_corpus_bigrams = X_bigrams.sum(axis=0)

print("Список признаков (униграмм):")
print(feature_names_unigrams,"\n")
print("Частота встречаемости униграмм в корпусе:")
print(count_in_corpus_unigrams,"\n")

print("Список признаков (биграмм):")
print(feature_names_bigrams,"\n")
print("Частота встречаемости биграмм в корпусе:")
print(count_in_corpus_bigrams,"\n")


Сохраним полученные данные в формате словарей для удобства обращения к ним. Биграммы разбиваем на слова и используем как кортежи в качестве ключей словаря.

In [ ]:
# Составим словарь биграмм с частотой встречаемости (для удобства)
# Используем уже полученные count_in_corpus_bigrams и feature_names_bigrams
# Преобразуем их в словарь для удобства
bigram_counts = {}
for i, bigram in enumerate(feature_names_bigrams):
    # count_in_corpus_bigrams - это разреженная матрица, нужно получить значение
    count = count_in_corpus_bigrams[0, i]
    if count > 0:
        w1, w2 = bigram.split()
        bigram_counts[(w1, w2)] = count

In [ ]:
# То же самое для униграмм
# Используем уже полученные count_in_corpus_unigrams и feature_names_unigrams
unigram_counts = {}
for i, unigram in enumerate(feature_names_unigrams):
     # count_in_corpus_unigrams - это разреженная матрица, нужно получить значение
    count = count_in_corpus_unigrams[0, i]
    if count > 0:
        unigram_counts[unigram] = count

Ниже представлена функция, которая принимает на вход текущее слово и предсказывает следующее слово, основываясь на значениях условных вероятностей появления следующего слова при условии наличия текущего.

Значения условных вероятностей мы вычисляем, основываясь на частоте появления биграмм с текущим словом на первом месте и частоте встречаемости текущего слова в корпусе
P(следующее слово | текущее слово) = P(текущее слово и следующее слово в той же последовательности) / P(текущее слово)


$
P(\text{следующее слово} \mid \text{текущее слово}) =
\frac{P(\text{текущее слово}, \text{следующее слово})}{P(\text{текущее слово})} = \frac{Count(\text{текущее слово}, \text{следующее слово})}{Count(\text{текущее слово})}
$


In [ ]:
# Реализация функции предсказания следующего слова по текущему
def predict_next_word(current_word, unigram_counts, bigram_counts):
    if current_word not in unigram_counts:
        return "Слово не найдено в корпусе"

    # Находим все биграммы, начинающиеся со слова current_word
    # Найденные биграммы (вместе с их частотами) сохраняются в словаре
    # possible_next_words, где ключ - это второе слово биграммы (возможное следующее слово),
    # а значение - частота этой биграммы.
    possible_next_words = {}
    for (w1, w2), count in bigram_counts.items():
        if w1 == current_word:
            possible_next_words[w2] = count

    if not possible_next_words:
        return "Нет биграмм, начинающихся с этого слова"

    # Вычисляем условные вероятности и находим слово с максимальной вероятностью из словаря possible_next_words
    best_next_word = None
    max_probability = -1

    for next_word, count in possible_next_words.items():
        # Вероятность P(next_word | current_word) = Count(current_word, next_word) / Count(current_word)
        # Убедимся, что unigram_counts[current_word] не равен нулю перед делением
        if unigram_counts[current_word] > 0:
            probability = count / unigram_counts[current_word]
            if probability > max_probability:
                max_probability = probability
                best_next_word = next_word

    return best_next_word



In [ ]:
# Демонстрация работы модели
print("--- Демонстрация предсказания ---")
# Используем слова из feature_names_unigrams для демонстрации
test_words = list(unigram_counts.keys())[:3] # Берем первые 3 слова из словаря
for word in test_words:
    prediction = predict_next_word(word, unigram_counts, bigram_counts)
    print(f"После слова '{word}' модель предсказывает: '{prediction}'")

### Обсуждение ограничений такой модели
Ограничения простой языковой модели, основанной на использовании N-грамм:
1. Проблема разреженности данных: Модель не может предсказать следующее слово, если соответствующая биграмма не встречалась в обучающем корпусе.
2. Не учитывает долгосрочные зависимости: Модель смотрит только на предыдущее слово, игнорируя более ранний контекст."
3. Чувствительна к шуму и ошибкам в тексте.
4. Ограниченный словарь: Не может работать со словами, которых нет в обучающем корпусе (OOV - Out-of-Vocabulary).
5. Не учитывает морфологию и синтаксис.

## Word2vec, FastText, GloVe

**Векторные представления слов (Word Embeddings)** — это плотные числовые векторы, которые представляют слова таким образом, чтобы слова с похожим значением имели схожие векторные представления (находились близко в многомерном векторном пространстве). Эти представления улавливают семантические и синтаксические отношения между словами.

В отличие от разреженных представлений, таких как Bag-of-Words или TF-IDF, где размерность вектора равна размеру словаря, а большинство значений нулевые, эмбеддинги обычно имеют гораздо меньшую фиксированную размерность и состоят из вещественных чисел.

### Word2Vec

**Word2Vec** — это одна из первых и наиболее известных моделей для обучения векторных представлений слов. Она использует простые нейронные сети для предсказания слов на основе их контекста. Существует две основные архитектуры Word2Vec:

*   **Skip-gram:** Модель предсказывает контекстные слова, находящиеся в окне вокруг данного целевого слова.
*   **CBOW (Continuous Bag-of-Words):** Модель предсказывает целевое слово на основе усредненных векторов контекстных слов в окне.

Word2Vec хорошо улавливает семантические отношения, такие как "король" - "мужчина" + "женщина" ≈ "королева".

### FastText

**FastText**, разработанный Facebook, является расширением Word2Vec. Ключевое отличие FastText в том, что он учитывает **символьные N-граммы** в словах. Вместо того чтобы представлять каждое слово как единицу, FastText представляет слово как набор его символьных N-грамм (в дополнение к самому слову). Это имеет несколько преимуществ:

*   **Обработка редких и внесловарных слов (OOV):** FastText может генерировать векторы для слов, которых нет в обучающем словаре, путем комбинирования векторов их символьных N-грамм.
*   **Лучшее представление морфологии:** Модель лучше улавливает сходство между морфологически связанными словами (например, "бежать", "бежит", "бегущий").

FastText часто показывает лучшие результаты на морфологически богатых языках.

### GloVe

**GloVe (Global Vectors for Word Representation)** — это модель, разработанная в Стэнфорде. В отличие от предсказательных моделей типа Word2Vec и FastText, GloVe основан на **глобальной статистике частоты совместного появления слов** в корпусе. Он обучается на матрице совместной встречаемости слов, пытаясь найти векторы слов таким образом, чтобы их скалярное произведение было связано с логарифмом частоты их совместного появления.

GloVe сочетает в себе преимущества методов, основанных на локальном контекстном окне (как Word2Vec), и методов, основанных на глобальной матричной факторизации.

### Работа с эмбеддингами

После обучения на большом корпусе текста, эмбеддинги можно использовать в качестве признаков для различных задач NLP, таких как классификация текста, определение сходства текстов, машинный перевод и другие. Предобученные эмбеддинги, обученные на огромных общедоступных корпусах (например, Wikipedia, Common Crawl), часто используются в качестве отправной точки и могут быть дообучены на специфичных для задачи данных.

### Задание 2: Работа с предобученными эмбеддингами GloVe

В этом задании мы поработаем с предобученными векторными представлениями слов, используя библиотеку `gensim`. Мы будем исследовать семантические связи между словами, закодированные в этих векторах.

1.  **Загрузка предобученных векторов:**
Мы используем модуль `gensim.downloader` для загрузки небольшого набора предобученных векторов английских слов, в частности `glove-wiki-gigaword-50` (50-мерные векторы, обученные на части Wikipedia). Обратите внимание, что это может занять некоторое время и требует подключения к интернету.

Вы также можете протестировать обученные модели

*   Word2Vec: "word2vec-google-news-300"
*   FastText: "fasttext-wiki-news-subwords-300"




In [ ]:
# Загрузка предобученных векторов
# Используем небольшой набор Glove
print("Загрузка модели 'glove-wiki-gigaword-50'...")
try:
    model = api.load("glove-wiki-gigaword-50")
    print("Модель успешно загружена.")
except Exception as e:
    print(f"Ошибка загрузки модели: {e}")
    print("Пожалуйста, проверьте подключение к интернету или выберите другую модель.")
    model = None

2.  **Получение векторов слов:** Из загруженной модели получим векторное представление для нескольких слов, например, "king", "woman", "man". Выведем полученные векторы.
3.  **Поиск похожих слов:** Для слова "king" найдём 10 слов, наиболее похожих с ним по смыслу, используя метод модели Glove `.most_similar()`.
4.  **Векторная арифметика:** Выполним операцию векторной арифметики для демонстрации семантических отношений: `вектор("король") - вектор("мужчина") + вектор("женщина")`. Мы получим слово, вектор которого наиболее близок к результату этой операции, используя метод `.most_similar()` с параметрами `positive` и `negative`.

In [ ]:
if model:
    # Получение векторов слов
    words_to_get_vectors = ["king", "woman", "man"]
    print("\n--- Векторы для заданных слов ---")
    for word in words_to_get_vectors:
        try:
            vector = model[word]
            print(f"Вектор для слова '{word}': {vector[:5]}...") # Выводим только первые 5 элементов для краткости
        except KeyError:
            print(f"Слово '{word}' не найдено в словаре модели.")

    # Поиск похожих слов
    words_to_find_similar = ["king"]
    print("\n--- Слова, наиболее похожие по смыслу ---")
    for word in words_to_find_similar:
        try:
            similar_words = model.most_similar(word, topn=10)
            print(f"Слова, похожие на '{word}':")
            for similar_word, score in similar_words:
                print(f"  {similar_word}: {score:.4f}")
        except KeyError:
            print(f"Слово '{word}' не найдено в словаре модели.")


    # Векторная арифметика: "король" - "man" + "woman" = ?
    print("\n--- Векторная арифметика: 'king' - 'man' + 'woman' ---")
    try:
        result_vector = model.most_similar(positive=['king', 'woman'], negative=['man'], topn=1)
        if result_vector:
            word_analogy, score_analogy = result_vector[0]
            print(f"Результат аналогии: '{word_analogy}' (сходство: {score_analogy:.4f})")
        else:
             print("Не удалось выполнить векторную арифметику.")
    except KeyError as e:
        print(f"Не удалось выполнить векторную арифметику. Одно из слов не найдено в словаре модели: {e}")
    except Exception as e:
         print(f"Произошла ошибка при выполнении векторной арифметики: {e}")

**Задание:**
1. Получите слова, наиболее похожие на слово 'paris'
2. Задайте с помощью векторной арифметики слово 'berlin', используя векторы слов 'paris', 'france', 'germany'.

In [ ]:

#### ВСТАВЬТЕ КОД СЮДА
...
####


Предобученные эмбеддинги могут использоваться:
1. Как входные признаки для моделей машинного обучения (например, для классификации текста).
2. Для инициализации слоев эмбеддингов в нейронных сетях (transfer learning).
3. Для поиска семантически похожих документов или запросов (путем усреднения векторов слов).
4. Для визуализации семантического пространства слов (с помощью техник снижения размерности, таких как t-SNE).
5. Для улучшения качества решений задач, требующих понимания семантики слов (NER, sentiment analysis).


## **Эмбеддинги для русскоязычных текстов** на примере библиотеки Navec

**Navec** — это библиотека, предоставляющая предобученные эмбеддинги для русского языка. Эти эмбеддинги были обучены на больших корпусах русских текстов и хорошо подходят для задач, связанных с русскоязычными данными. Navec основана на модели FastText, что означает, что она также учитывает символьные N-граммы и может работать с редкими и внесловарными словами.

Использование предобученных эмбеддингов, таких как Navec, позволяет избежать необходимости обучать эмбеддинги с нуля на собственном (возможно, небольшом) корпусе, что экономит время и вычислительные ресурсы, а также часто дает лучшие результаты, так как модель уже "видела" большое количество разнообразного текста.

Основные шаги при работе с Navec:
1.  **Установка библиотеки:** Библиотека устанавливается через pip.
2.  **Загрузка модели:** Модель можно загрузить по ссылке.
3.  **Работа с векторами:** Получение векторов для слов и использование стандартных операций с векторами (поиск похожих слов, векторная арифметика), аналогично работе с другими библиотеками эмбеддингов.

In [ ]:
# Загрузка предобученной модели
# Ссылка на модель

url = 'https://storage.yandexcloud.net/natasha-navec/packs/navec_hudlit_v1_12B_500K_300d_100q.tar'
model_path = 'navec_model.tar'

print(f"Загрузка модели с {url}...")
try:
    response = requests.get(url, stream=True)
    response.raise_for_status() # Проверка на ошибки HTTP
    with open(model_path, 'wb') as f:
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)
    print("Модель успешно загружена.")

    # Загрузка модели из файла
    navec = Navec.load(model_path)
    print("Модель загружена в память.")

except requests.exceptions.RequestException as e:
    print(f"Ошибка загрузки файла: {e}")
    navec = None
except Exception as e:
    print(f"Ошибка при загрузке или обработке модели : {e}")
    navec = None

In [ ]:
if navec:
    # 2. Получение векторов слов
    words_to_get_vectors = ["король", "женщина", "мужчина", "париж", "франция", "германия"]
    print("\n--- Векторы для заданных слов ---")
    for word in words_to_get_vectors:
        if word in navec:
            vector = navec[word]
            print(f"Вектор для слова '{word}': {vector[:5]}...") # Выводим только первые 5 элементов
        else:
            print(f"Слово '{word}' не найдено в словаре модели.")

    # Продемонстрируем получение вектора
    print("\n--- Пример получения вектора и проверки наличия слова ---")
    test_word = "данные"
    if test_word in navec:
        print(f"Слово '{test_word}' найдено в словаре. Вектор: {navec[test_word][:5]}...")
    else:
        print(f"Слово '{test_word}' не найдено в словаре.")

else:
    print("\nНе удалось загрузить модель.")

## Косинусное сходство/расстояние

Для поиска похожих слов можно использовать поиск по косинусному сходству. Чем выше косинусное сходство, тем ближе векторизованные предстваления слов расположены друг к другу в пространстве эмбеддингов.

косинусное расстояние = 1 - косинусное сходство

(cosine distance = 1 - cosine similarity)

In [ ]:
# Пример: вычисление косинусного сходства между словами

# Проверяем наличие слов перед использованием
def cosine_similarity(word1, word2):
    if word1 in navec and word2 in navec:
        word1_vec = navec[word1]
        word2_vec = navec[word2]
        similarity_navec = 1 - cosine(word1_vec, word2_vec)
        print(f"\nКосинусное сходство между '{word1}' и '{word2}': {similarity_navec}")
    else:
        print(f"\nОдно или оба слова ('{word1}', '{word2}') не найдены в словаре NaveC.")
    return

In [ ]:
cosine_similarity('кошка', 'собака')

cosine_similarity('врач', 'доктор')

cosine_similarity('кошка', 'ложка')

cosine_similarity('ложка', 'вилка')

cosine_similarity('утюг', 'водопад')

# Попробуйте добавить свои примеры и вы увидите, что не всегда возможно предугадать, насколько близко расположены эмбеддинги двух слов

## Основы нейронных сетей - градиентный спуск

Нейронные сети — это вычислительные модели, которые состоят из взаимосвязанных узлов (нейронов), организованных слоями.

### Основные компоненты нейронной сети

*   **Входной слой:** Получает исходные данные (например, векторизованный текст или эмбеддинги).
*   **Скрытые слои:** Промежуточные слои, где происходят основные вычисления. В глубоких нейронных сетях таких слоев несколько.
*   **Выходной слой:** Формирует конечный результат (например, вероятность принадлежности текста к определенному классу).
*   **Нейроны:** Базовые вычислительные единицы, которые получают входные сигналы, применяют к ним взвешенную сумму и передают результат через функцию активации.
*   **Веса и смещения:** Параметры модели, которые подстраиваются в процессе обучения. Веса определяют силу связи между нейронами, а смещения сдвигают выходное значение нейрона.

### Функции активации

Функция активации определяет выход нейрона на основе его входных данных. Она вносит нелинейность в модель, что позволяет нейронным сетям обучаться сложным зависимостям в данных. Некоторые распространенные функции активации:

*   **Сигмоида (Sigmoid):** Сжимает входное значение в диапазон от 0 до 1. Часто используется в выходных слоях для задач бинарной классификации.
*   **Гиперболический тангенс (Tanh):** Сжимает входное значение в диапазон от -1 до 1. Похожа на сигмоиду, но центрирована около нуля.
*   **ReLU (Rectified Linear Unit):** Возвращает входное значение, если оно положительное, и 0 в противном случае (max(0, x)). Является одной из самых популярных функций активации в скрытых слоях благодаря своей простоте и способности решать проблему "затухающих градиентов".
*   **Softmax:** Преобразует вектор входных значений в распределение вероятностей. Обычно используется в выходном слое для задач многоклассовой классификации.

### Функции потерь (Loss Functions)

Функция потерь измеряет, насколько хорошо модель справляется с задачей, сравнивая предсказанные значения с фактическими. Цель обучения нейронной сети — минимизировать эту функцию. Примеры функций потерь:

*   **Mean Squared Error (MSE):** Используется для задач регрессии.
*   **Cross-Entropy (Кросс-энтропия):** Используется для задач классификации. `Binary Cross-Entropy` для бинарной классификации и `Categorical Cross-Entropy` для многоклассовой.

### Градиентный спуск и его модификации

**Градиентный спуск** — это основной алгоритм оптимизации, используемый для обучения нейронных сетей. Он итеративно корректирует веса и смещения модели в направлении, противоположном градиенту функции потерь (направлению наибольшего возрастания функции). Это позволяет постепенно уменьшать ошибку модели.

**Скорость обучения (Learning Rate):** Параметр, определяющий размер шага при корректировке весов во время градиентного спуска. Слишком большая скорость может привести к перескакиванию минимума, а слишком маленькая — к очень медленному обучению.

**Проблемы стандартного градиентного спуска:**
*   Может застрять в локальных минимумах (хотя в многомерном пространстве нейронных сетей это не такая большая проблема, как сходимость в седловых точках).
*   Медленно сходится на плоских участках функции потерь.
*   Требует вычисления градиента по всему набору данных на каждой итерации (Batch Gradient Descent), что может быть вычислительно затратно для больших данных.

**Модификации градиентного спуска:**

*   **Stochastic Gradient Descent (SGD):** На каждой итерации обновляет веса на основе градиента, вычисленного по одному случайно выбранному примеру данных. Это делает процесс обучения более быстрым и позволяет избежать застревания в некоторых локальных минимумах, но обновления весов более шумные.
*   **Mini-Batch Gradient Descent:** Компромисс между Batch GD и SGD. Использует мини-батчи (небольшие случайные подмножества данных) для вычисления градиента и обновления весов. Наиболее часто используемый метод на практике.
*   **Momentum:** Добавляет "импульс" к обновлению весов, помогая алгоритму преодолевать локальные минимумы и быстрее сходиться, особенно на плоских участках. Обновление весов зависит не только от текущего градиента, но и от градиентов на предыдущих шагах.
*   **Adagrad (Adaptive Gradient):** Адаптирует скорость обучения для каждого параметра модели индивидуально, уменьшая скорость для часто обновляемых параметров и увеличивая для редких. Может сильно уменьшать скорость обучения со временем.
*   **RMSprop (Root Mean Square Propagation):** Похож на Adagrad, но использует скользящее среднее квадратов градиентов, что помогает избежать чрезмерного уменьшения скорости обучения.
*   **Adam (Adaptive Moment Estimation):** Объединяет идеи Momentum и RMSprop. Адаптирует скорость обучения для каждого параметра и использует экспоненциально затухающие средние градиентов и их квадратов. Один из самых популярных и эффективных оптимизаторов на сегодняшний день.

### Градиентный спуск для функции одной переменной

Найдем локальный минимум функции $f(x)=x^3-5x+2$ методом градиентного спуска.

In [ ]:
def f(x):
   return x**3 - 5*x + 2

x_values = [x for x in np.arange(-5, 5, 0.1)]
f_values = [f(x) for x in np.arange(-5, 5, 0.1)]

plt.figure(figsize=(8,8))

plt.axvline(x=0, c = 'black')
plt.axhline(y=0, c = 'black')
plt.plot(x_values, f_values);

Вычислим производную функции $f(x)$:

$f'(x)=3x^2-5$.


In [ ]:
# Производная
def grad_f (x):
    return 3*(x**2) - 5

In [ ]:
def gradient_descent(x_start, learning_rate, epsilon, num_iterations):
    x_curr = x_start

    trace = []
    trace.append(x_curr)

    for i in range(num_iterations):
      x_new = x_curr - learning_rate * grad_f(x_curr)
      trace.append(x_new)

      if abs(x_new - x_curr) < epsilon:
        return x_curr, trace

      x_curr = x_new

    return x_curr, trace

Попробуем стартовать из точки $x_0 = 4$ и идти с маленьким шагом $\eta = 0.01$.


In [ ]:
x0 = 4
eta = 0.01
epsilon = 0.001
iterations = 100

Визуализируем процесс поиска минимума (красные точки) и найденный минимум (зеленая точка).

In [ ]:
xmin, trace = gradient_descent(x0, eta, epsilon, iterations)

x_values = [x for x in np.arange(-5, 5, 0.1)]
f_values = [f(x) for x in x_values]

plt.figure(figsize=(10,10))

plt.axvline(x=0, c = 'black')
plt.axhline(y=0, c = 'black')

plt.plot(x_values, f_values)

plt.xlim([-3, 5])
plt.ylim([-5, 7])

plt.title('График функции f(x) - градиентный спуск')

plt.xlabel('x')
plt.ylabel('f(x)')

trace_values = [f(x) for x in trace]
plt.scatter(trace, trace_values, c='red')
plt.scatter([xmin],[f(xmin)], c='green')

plt.show()

In [ ]:
xmin

### Задание 3: Градиентный спуск

При помощи градиентного спуска найдите минимум функции
$f(x) = x^3 - 3x^2 + 4$.

In [ ]:
#### ВСТАВЬТЕ КОД СЮДА
...
####